# Build Canonical `df_combined`

This notebook rebuilds the final thesis corpus exactly from the cleaned outlet CSVs, using the same logic that historically built `overall_df` in `07_OverallTM.ipynb`.

It is now the notebook-level source of truth for:
- how the combined corpus is created,
- which cleaned files go into it,
- how `row_id` is assigned,
- and the overview tables/plots used to document the corpus.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "00_Initial EDA"
CANONICAL_PATH = DATA_DIR / "df_combined.csv"
REQUIRED_COLUMNS = ["Date", "Title", "Text", "source"]
OUTLET_SPECS = [
    {"key": "antispiegel", "source": "Antispiegel", "filename": "antispiegel_clean.csv", "berlin_tz": False},
    {"key": "compact", "source": "Compact", "filename": "compact_clean.csv", "berlin_tz": False},
    {"key": "nius", "source": "Nius", "filename": "nius_clean.csv", "berlin_tz": False},
    {"key": "rt_de", "source": "RT_de", "filename": "rt_de_clean.csv", "berlin_tz": False},
    {"key": "tichys", "source": "Tichys_Einblick", "filename": "tichys_clean.csv", "berlin_tz": False},
    {"key": "dkurier", "source": "Deutschlandkurier", "filename": "dkurier_clean.csv", "berlin_tz": False},
    {"key": "tagesschau", "source": "Tagesschau", "filename": "tagesschau_clean.csv", "berlin_tz": True},
]

CANONICAL_PATH

In [ ]:
def load_clean_outlet_csv(spec: dict) -> tuple[pd.DataFrame, Path]:
    csv_path = DATA_DIR / spec["filename"]
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing cleaned outlet CSV: {csv_path}")

    frame = pd.read_csv(csv_path)
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise KeyError(f"{csv_path.name} is missing required columns: {missing_columns}")

    frame = frame[REQUIRED_COLUMNS].copy()
    parsed_dates = pd.to_datetime(frame["Date"], errors="coerce", utc=True)
    if spec["berlin_tz"]:
        parsed_dates = parsed_dates.dt.tz_convert("Europe/Berlin")
    frame["Date"] = parsed_dates.dt.tz_localize(None).dt.normalize()
    return frame, csv_path


def build_df_combined() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    datasets: dict[str, pd.DataFrame] = {}
    for spec in OUTLET_SPECS:
        frame, csv_path = load_clean_outlet_csv(spec)
        datasets[spec["key"]] = frame
        print(f"{spec['source']:<18} {frame.shape} <- {csv_path}")

    overall_df = pd.concat(datasets.values(), ignore_index=True)
    overall_df = overall_df.reset_index(drop=True)
    overall_df["row_id"] = overall_df.index + 1
    return datasets, overall_df


def compare_dataframes_as_strings(left: pd.DataFrame, right: pd.DataFrame) -> list[str]:
    if left.columns.tolist() != right.columns.tolist():
        return ["__columns__"]
    if left.shape != right.shape:
        return ["__shape__"]

    mismatches = []
    for column in left.columns:
        left_values = left[column].astype("string").fillna("<NA>")
        right_values = right[column].astype("string").fillna("<NA>")
        if not left_values.equals(right_values):
            mismatches.append(column)
    return mismatches

In [ ]:
datasets, overall_df = build_df_combined()
existing_df = pd.read_csv(CANONICAL_PATH) if CANONICAL_PATH.exists() else pd.DataFrame()

print(f"Rebuilt shape: {overall_df.shape}")
print(f"row_id unique: {overall_df['row_id'].is_unique}")

if not existing_df.empty:
    print(f"Existing shape: {existing_df.shape}")
    mismatch_columns = compare_dataframes_as_strings(overall_df, existing_df)
    print(f"Mismatch columns: {mismatch_columns}")
    if mismatch_columns:
        raise AssertionError(
            "Rebuilt dataframe does not match the existing canonical df_combined.csv. "
            f"Mismatched columns: {mismatch_columns}"
        )

overall_df.head()

In [ ]:
source_counts = overall_df["source"].value_counts().rename_axis("source").reset_index(name="articles")
source_counts

In [ ]:
overall_dates = pd.to_datetime(overall_df["Date"], errors="coerce")
overall_dates.min(), overall_dates.max()


In [ ]:
tmp = overall_df.copy()
tmp["Date_parsed"] = pd.to_datetime(tmp["Date"], errors="coerce", utc=True)

qc = (
    tmp.groupby("source", dropna=False)
       .agg(
           rows=("Date", "size"),
           parsed_ok=("Date_parsed", lambda s: s.notna().sum()),
           parsed_na=("Date_parsed", lambda s: s.isna().sum()),
           parse_rate=("Date_parsed", lambda s: s.notna().mean()),
           min_date=("Date_parsed", "min"),
           max_date=("Date_parsed", "max"),
       )
       .sort_values("rows", ascending=False)
)

qc

In [ ]:
df_plot = overall_df.copy()
df_plot["Date"] = pd.to_datetime(df_plot["Date"], errors="coerce", utc=True).dt.tz_localize(None)
df_plot = df_plot.dropna(subset=["Date", "source"])

freq = "M"
count_df = (
    df_plot
    .assign(period=df_plot["Date"].dt.to_period(freq).dt.to_timestamp())
    .groupby(["period", "source"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

ax = count_df.plot.area(figsize=(14, 7), alpha=0.9)
ax.set_title("Documents per Source Over Time")
ax.set_xlabel("Time")
ax.set_ylabel("Number of Documents")
ax.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

count_df

In [ ]:
rebuilt_tagesschau_preview = datasets["tagesschau"][["Date", "Title"]].head(10)
rebuilt_tagesschau_preview

In [ ]:
overall_df.to_csv(CANONICAL_PATH, index=False)
print(f"Wrote canonical combined corpus to: {CANONICAL_PATH}")